# TNT hybrid V3 — post-pretraining optimization

This notebook reuses the completed M1-pretrained V3 checkpoint. It does **not** repeat M1 pretraining and never evaluates Gold-test.

The workflow is deliberately sequential: first test inference-only boundary and word-gate controls on Gold-dev; then, if useful, run one or two short five-epoch Gold adaptations from the same immutable M1 checkpoint. Short adaptations select checkpoints by generated **WER**, because the previous CER-selected Gold stage damaged word-level behavior.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').is_dir():
    candidates = [parent for parent in PROJECT_ROOT.parents if (parent / 'src').is_dir()]
    if not candidates:
        raise FileNotFoundError('Run from the PFE repository or a subdirectory.')
    PROJECT_ROOT = candidates[0]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import torch
from IPython.display import display

from src.modeling.tnt_hybrid_v3_optimization import (
    INFERENCE_DIR, INFERENCE_VARIANTS, RESULTS_DIR,
    completed_m1_pretraining, gold_adaptation_summary,
    load_optimization_data, run_inference_ablation,
    run_short_gold_adaptation,
)

gold = load_optimization_data()
m1_checkpoint, m1_manifest = completed_m1_pretraining()
print('M1 checkpoint:', m1_checkpoint.resolve())
print('Gold rows:', {split: len(frame) for split, frame in gold.items()})
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
print('Results:', RESULTS_DIR.resolve())

c:\Users\thioy\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\cuda\__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


M1 checkpoint: C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_v3_m1_gate_v1\m1\seed_2026\synthetic_pretrain\attempt_001\best_model
Gold rows: {'train': 142, 'dev': 30, 'test': 29}
CUDA: True NVIDIA GeForce RTX 5060 Laptop GPU
Results: C:\Users\thioy\Desktop\School\PFE\results\m7_seq2seq\tnt_hybrid_v3_optimization_v1


## 1. Fast inference ablation

These variants reuse identical weights. They only change whether boundary/word decisions are accepted and their confidence thresholds. This should take seconds or a few minutes, not hours. Results and predictions are saved under the optimization results directory.

In [2]:
display(pd.DataFrame([variant.__dict__ for variant in INFERENCE_VARIANTS]))

,name,use_boundary_inference,boundary_confidence_threshold,use_word_keep_gate,word_keep_threshold,use_hard_lexical_protection
0,default,True,0.5,True,0.70,True
1,no_boundary,False,0.5,True,0.70,True
2,boundary_t070,True,0.7,True,0.70,True
3,boundary_t080,True,0.8,True,0.70,True
4,no_word_gate,True,0.5,False,0.70,True
5,word_keep_t050,True,0.5,True,0.50,True
6,word_keep_t085,True,0.5,True,0.85,True
7,no_boundary_word_t050,False,0.5,True,0.50,True


In [3]:
RUN_INFERENCE_ABLATION = False  # results already exist; set True to recompute

if RUN_INFERENCE_ABLATION:
    inference_summary = run_inference_ablation(
        gold['dev'], checkpoint=m1_checkpoint
    )
else:
    summary_path = INFERENCE_DIR / 'summary.csv'
    inference_summary = pd.read_csv(summary_path) if summary_path.is_file() else pd.DataFrame()
display(inference_summary.sort_values(['wer', 'cer']) if not inference_summary.empty else inference_summary)

,variant,name,use_boundary_inference,boundary_confidence_threshold,use_word_keep_gate,word_keep_threshold,use_hard_lexical_protection,rows,cer,wer,...,correction_fn,correction_precision,correction_recall,correction_f1,overcorrection_rate,unchanged_reference_rows,unchanged_sentence_overcorrection_rate,changed_sentence_rows,inference_seconds,predictions_path
5,word_keep_t050,word_keep_t050,True,0.5,True,0.50,True,30,0.187779,0.583051,...,70,0.028169,0.027778,0.027972,0.971831,0,0.0,30,0.018513,C:\Users\thioy\Desktop\School\PFE\results\m7_s...
7,no_boundary_word_t050,no_boundary_word_t050,False,0.5,True,0.50,True,30,0.187779,0.583051,...,70,0.028169,0.027778,0.027972,0.971831,0,0.0,30,0.017183,C:\Users\thioy\Desktop\School\PFE\results\m7_s...
0,default,default,True,0.5,True,0.70,True,30,0.193741,0.593220,...,69,0.039474,0.041667,0.040541,0.960526,0,0.0,30,0.348575,C:\Users\thioy\Desktop\School\PFE\results\m7_s...
1,no_boundary,no_boundary,False,0.5,True,0.70,True,30,0.193741,0.593220,...,69,0.039474,0.041667,0.040541,0.960526,0,0.0,30,0.022617,C:\Users\thioy\Desktop\School\PFE\results\m7_s...
2,boundary_t070,boundary_t070,True,0.7,True,0.70,True,30,0.193741,0.593220,...,69,0.039474,0.041667,0.040541,0.960526,0,0.0,30,0.021165,C:\Users\thioy\Desktop\School\PFE\results\m7_s...
3,boundary_t080,boundary_t080,True,0.8,True,0.70,True,30,0.193741,0.593220,...,69,0.039474,0.041667,0.040541,0.960526,0,0.0,30,0.018677,C:\Users\thioy\Desktop\School\PFE\results\m7_s...
6,word_keep_t085,word_keep_t085,True,0.5,True,0.85,True,30,0.197466,0.603390,...,69,0.040541,0.041667,0.041096,0.959459,0,0.0,30,0.019125,C:\Users\thioy\Desktop\School\PFE\results\m7_s...
4,no_word_gate,no_word_gate,True,0.5,False,0.70,True,30,0.198957,0.610169,...,69,0.041667,0.041667,0.041667,0.958333,0,0.0,30,0.019323,C:\Users\thioy\Desktop\School\PFE\results\m7_s...


### Inspect qualitative differences

Choose a candidate only after checking both metrics and malformed outputs. The completed ablation found that boundary variants were identical. `word_keep_t050` gave the best CER/WER/chrF and reduced false edits from 73 to 69, but correction F1 fell from 0.0405 to 0.0280 because one true correction was also suppressed. It is therefore an optimization candidate, not an unqualified winner.

In [4]:
VARIANTS_TO_COMPARE = ['default', 'no_boundary', 'boundary_t070', 'word_keep_t050']
comparison = gold['dev'][['source_id', 'source_text', 'target_text']].rename(columns={'target_text': 'reference'})
for variant_name in VARIANTS_TO_COMPARE:
    path = INFERENCE_DIR / f'{variant_name}_predictions.csv'
    if path.is_file():
        predictions = pd.read_csv(path, keep_default_na=False)
        comparison = comparison.merge(
            predictions[['source_id', 'prediction']].rename(columns={'prediction': f'prediction_{variant_name}'}),
            on='source_id', validate='one_to_one',
        )
display(comparison)

,source_id,source_text,reference,prediction_default,prediction_no_boundary,prediction_boundary_t070,prediction_word_keep_t050
0,gold:dev:10000,​ té dou wakhé dank,té du waxee ndànk,H te du waxe dank,H te du waxe dank,H te du waxe dank,H te du waxe dank
1,gold:dev:10003,cachet wam mo wara diekh. sokhna ak salimata l...,cachet wam moo wara jeex sokhna ak salimata lé...,Caceet wam moo wara jeex. sokhna ak salimata l...,Caceet wam moo wara jeex. sokhna ak salimata l...,Caceet wam moo wara jeex. sokhna ak salimata l...,Caceet wam moo wara jeex. sokhna ak salimata l...
2,gold:dev:10005,nimako diépé sakh da eupeu,ni ma ko jéppe sax da ëpp,Nimakoo jepe sax da ëpë,Nimakoo jepe sax da ëpë,Nimakoo jepe sax da ëpë,Nimakoo jepe sax da ëpë
3,gold:dev:10007,maman oumy mome rk lay bagna guis,maman oumy moom rek laay baña gis,maman oumy moome rk laay baña gis,maman oumy moome rk laay baña gis,maman oumy moome rk laay baña gis,maman oumy moome rk lay baña gis
4,gold:dev:10008,mére bi mo sof ba paré reuy lamign,mère bi moo soof ba pare réy làmmiñ,mére bi moo soof ba pare rëy laamign,mére bi moo soof ba pare rëy laamign,mére bi moo soof ba pare rëy laamign,mére bi moo sof ba pare rëy laamign
5,gold:dev:10013,franchement khady thiam nak dafa eupeul fi kou...,franchement khady thiam nak dafa ëppal fi kuy ...,Franceement khady thiam nak dafa ëpël fi kuy s...,Franceement khady thiam nak dafa ëpël fi kuy s...,Franceement khady thiam nak dafa ëpël fi kuy s...,Franceement khady thiam nak dafa ëpël fi kuy s...
6,gold:dev:10014,loutakh pere ndiouga di raamal vieux?alors que...,lu tax pere ndiouga di raamal vieux alors que ...,Lutax pere ndiouga di raamaal véëx?aaloors qëe...,Lutax pere ndiouga di raamaal véëx?aaloors qëe...,Lutax pere ndiouga di raamaal véëx?aaloors qëe...,Lutax pere ndiouga di raamal véëx?aaloors qëe ...
7,gold:dev:10020,ki di way baabel voix bi nèx na deh kanla,ki di woy baabel voix bi neex na de kan la,Ki di way baabeel voix bi nèx na dex kanlaa,Ki di way baabeel voix bi nèx na dex kanlaa,Ki di way baabeel voix bi nèx na dex kanlaa,Ki di way baabeel voix bi nèx na dex kanlaa
8,gold:dev:10021,maman omou kiye rayé si série douko ray,maman omou kuy reye ci série du ko rey,maman omou kiyée raye si série duko raay,maman omou kiyée raye si série duko raay,maman omou kiyée raye si série duko raay,maman omou kiye raye si série duko ray
9,gold:dev:10026,le meilleurs séries kou d'accord lalal j'aime ️,les meilleurs séries ku d'accord laalal j'aime,Loe meillërs seeriees ku d'accord laalaal j'ai...,Loe meillërs seeriees ku d'accord laalaal j'ai...,Loe meillërs seeriees ku d'accord laalaal j'ai...,Loe meillërs seeriees ku d'accord laalaal j'ai...


## 2. Optional short Gold adaptations

After the inference table is reviewed, set `SELECTED_INFERENCE_VARIANT` to the preferred reconstruction setting. The two training cells are independent and disabled by default. Both start from the same completed M1 checkpoint, run only five Gold epochs, and select by Gold-dev WER. `RERUN=False` resumes an interrupted matching run or reuses a completed one.

In [5]:
SELECTED_INFERENCE_VARIANT = 'word_keep_t050'
RERUN = False
RUN_GOLD_ADAPT_2E5 = True

if RUN_GOLD_ADAPT_2E5:
    record_2e5 = run_short_gold_adaptation(
        'lr_2e5', gold,
        inference_variant_name=SELECTED_INFERENCE_VARIANT,
        learning_rate=2e-5, epochs=5, rerun=RERUN,
    )
    display(pd.DataFrame([record_2e5.get('development_metrics', {})]))
else:
    print('2e-5 adaptation disabled')

Loading weights:   0%|          | 0/51 [00:00<?, ?it/s]

You are using a model of type `wolof_tnt_hybrid_v3_transformer` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


[M7] START m1_v3 / gold_adapt_lr_2e5_word_keep_t050 / seed 2026 / attempt 001 (trained)
[M7] Output: C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_v3_optimization_v1\m1_v3\seed_2026\gold_adapt_lr_2e5_word_keep_t050\attempt_001
[M7] Loading C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_v3_optimization_v1\adaptation_initializations\word_keep_t050_e6d87bff064f


You are using a model of type `wolof_tnt_hybrid_v3_transformer` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


...

Model weights saved in C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_v3_optimization_v1\m1_v3\seed_2026\gold_adapt_lr_2e5_word_keep_t050\attempt_001\best_model\model.safetensors
tokenizer config file saved in C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_v3_optimization_v1\m1_v3\seed_2026\gold_adapt_lr_2e5_word_keep_t050\attempt_001\best_model\tokenizer_config.json
tokenizer config file saved in C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_v3_optimization_v1\m1_v3\seed_2026\gold_adapt_lr_2e5_word_keep_t050\attempt_001\best_model\tokenizer_config.json


[M7] Running development prediction and final metrics


The following columns in the test set don't have a corresponding argument in `TntHybridV3TransformerForConditionalGeneration.forward` and have been ignored: target_text, source_text, source_id. If target_text, source_text, source_id are not expected by `TntHybridV3TransformerForConditionalGeneration.forward`,  you can safely ignore this message.

***** Running Prediction *****
  Num examples = 30
  Batch size = 64


[M7] COMPLETE m1_v3 / gold_adapt_lr_2e5_word_keep_t050 / seed 2026 / attempt 001; dev CER=0.1780923994038748


,rows,cer,wer,chrf,exact_match,correction_tp,correction_fp,correction_fn,correction_precision,correction_recall,correction_f1,overcorrection_rate,unchanged_reference_rows,unchanged_sentence_overcorrection_rate
0,30,0.178092,0.579661,62.531777,0.0,3,68,69,0.042254,0.041667,0.041958,0.957746,0,0.0


In [8]:
RUN_GOLD_ADAPT_5E5 = True

if RUN_GOLD_ADAPT_5E5:
    record_5e5 = run_short_gold_adaptation(
        'lr_5e5', gold,
        inference_variant_name=SELECTED_INFERENCE_VARIANT,
        learning_rate=5e-5, epochs=5, rerun=RERUN,
    )
    display(pd.DataFrame([record_5e5.get('development_metrics', {})]))
else:
    print('5e-5 adaptation disabled')


[M7] START m1_v3 / gold_adapt_lr_5e5_word_keep_t050 / seed 2026 / attempt 001 (trained)
[M7] Output: C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_v3_optimization_v1\m1_v3\seed_2026\gold_adapt_lr_5e5_word_keep_t050\attempt_001
[M7] Loading C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_v3_optimization_v1\adaptation_initializations\word_keep_t050_e6d87bff064f


loading configuration file C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_v3_optimization_v1\adaptation_initializations\word_keep_t050_e6d87bff064f\config.json
loading configuration file C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_v3_optimization_v1\adaptation_initializations\word_keep_t050_e6d87bff064f\config.json
You are using a model of type `wolof_tnt_hybrid_v3_transformer` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.
Model config PreTrainedConfig {
  "architectures": [
    "TntHybridV3TransformerForConditionalGeneration"
  ],
  "attention_heads": 4,
  "bos_token_id": 1,
  "boundary_class_weights": [
    0.5533591001386836,
    1.2271996237756158,
    4.0,
   

Loading weights:   0%|          | 0/51 [00:00<?, ?it/s]

loading configuration file C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_v3_optimization_v1\adaptation_initializations\word_keep_t050_e6d87bff064f\generation_config.json
Generate config GenerationConfig {
  "assistant_confidence_threshold": 0.4,
  "assistant_lookbehind": 10,
  "bos_token_id": 1,
  "decoder_start_token_id": 1,
  "diversity_penalty": 0.0,
  "do_sample": false,
  "early_stopping": false,
  "encoder_no_repeat_ngram_size": 0,
  "encoder_repetition_penalty": 1.0,
  "eos_token_id": [
    2
  ],
  "epsilon_cutoff": 0.0,
  "eta_cutoff": 0.0,
  "length_penalty": 1.0,
  "max_length": 20,
  "min_length": 0,
  "no_repeat_ngram_size": 0,
  "num_assistant_tokens": 20,
  "num_assistant_tokens_schedule": "constant",
  "num_beam_groups": 1,
  "num_beams": 1,
  "num_return_sequences": 1,
  "output_attentions": false,
  "output_hidden_states": false,
  "output_scores": false,
  "pad_token_id": 0,
  "remove_invalid_values": false,
  "repetition_penalty": 1.0,
  "retur

[M7] Loaded 1,783,707 total parameters; training 1,783,707 (100.0000%) via full_finetune
[M7] Tokenizing 142 training and 30 development pairs


...

Model weights saved in C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_v3_optimization_v1\m1_v3\seed_2026\gold_adapt_lr_5e5_word_keep_t050\attempt_001\best_model\model.safetensors
tokenizer config file saved in C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_v3_optimization_v1\m1_v3\seed_2026\gold_adapt_lr_5e5_word_keep_t050\attempt_001\best_model\tokenizer_config.json
tokenizer config file saved in C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_v3_optimization_v1\m1_v3\seed_2026\gold_adapt_lr_5e5_word_keep_t050\attempt_001\best_model\tokenizer_config.json


[M7] Running development prediction and final metrics


The following columns in the test set don't have a corresponding argument in `TntHybridV3TransformerForConditionalGeneration.forward` and have been ignored: target_text, source_text, source_id. If target_text, source_text, source_id are not expected by `TntHybridV3TransformerForConditionalGeneration.forward`,  you can safely ignore this message.

***** Running Prediction *****
  Num examples = 30
  Batch size = 64


[M7] COMPLETE m1_v3 / gold_adapt_lr_5e5_word_keep_t050 / seed 2026 / attempt 001; dev CER=0.1706408345752608


,rows,cer,wer,chrf,exact_match,correction_tp,correction_fp,correction_fn,correction_precision,correction_recall,correction_f1,overcorrection_rate,unchanged_reference_rows,unchanged_sentence_overcorrection_rate
0,30,0.170641,0.572881,64.629214,0.0,3,57,69,0.05,0.041667,0.045455,0.95,0,0.0


## 3. Adaptation comparison

The untouched M1-pretrained checkpoint is always included as the control. Keep a Gold adaptation only if it improves WER without an unacceptable CER, correction-F1, or overcorrection trade-off.

In [9]:
adaptation_summary = gold_adaptation_summary()
display(adaptation_summary.sort_values(['wer', 'cer']))

,condition,learning_rate,epochs,rows,cer,wer,chrf,exact_match,correction_tp,correction_fp,correction_fn,correction_precision,correction_recall,correction_f1,overcorrection_rate,unchanged_reference_rows,unchanged_sentence_overcorrection_rate,checkpoint
2,gold_adapt_lr_5e5_word_keep_t050,0.00005,5.0,30,0.170641,0.572881,64.629214,0.0,3,57,69,0.050000,0.041667,0.045455,0.950000,0,0.0,C:\Users\thioy\Desktop\School\PFE\checkpoints\...
1,gold_adapt_lr_2e5_word_keep_t050,0.00002,5.0,30,0.178092,0.579661,62.531777,0.0,3,68,69,0.042254,0.041667,0.041958,0.957746,0,0.0,C:\Users\thioy\Desktop\School\PFE\checkpoints\...
0,m1_pretrained_no_gold_adaptation,0.00000,0.0,30,0.193741,0.593220,60.271611,0.0,3,73,69,0.039474,0.041667,0.040541,0.960526,0,0.0,C:\Users\thioy\Desktop\School\PFE\checkpoints\...


## 4. Locked Gold-test result (read-only)

The `5e-5`, epoch-3, word-threshold-0.50 candidate was locked before a one-time Gold-test evaluation. This cell only reads the frozen comparison; it does not run inference again or permit reselection.

In [ ]:
frozen_comparison_path = RESULTS_DIR / 'frozen_test' / 'test_comparison.csv'
if frozen_comparison_path.is_file():
    frozen_test_summary = pd.read_csv(frozen_comparison_path, keep_default_na=False)
    display(frozen_test_summary)
else:
    print('No frozen V3 test result exists.')